In [ ]:
import uuid
import pandas as pd
import psycopg2
from qdrant_client import QdrantClient
from qdrant_client.http import models
from sentence_transformers import SentenceTransformer
from final_project.connect_to_google_drive import get_sheets_client, SHEET_ID

c:\Users\Olena\Desktop\ucu\four\rag_support_system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Olena\Desktop\ucu\four\rag_support_system\.venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.7) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
COLLECTION_NAME = "ucu_documents_baseline"

In [3]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 236.69it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def get_text_from_postgres(file_id):
    try:
        conn = psycopg2.connect(
            dbname="ucu_rag_db", user="user", password="password", host="127.0.0.1"
        )
        cur = conn.cursor()
        cur.execute(
            "SELECT markdown_content FROM processed_documents WHERE google_drive_id = %s", 
            (file_id,)
        )
        result = cur.fetchone()
        cur.close()
        conn.close()
        return result[0] if result else None
    except Exception as e:
        print(f"ERROR: {e}")
        return None

In [5]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

In [7]:
sheets_client = get_sheets_client()
sheet = sheets_client.open_by_key(SHEET_ID).sheet1
df = pd.DataFrame(sheet.get_all_records())

In [8]:
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
    )

In [9]:
to_sync = df[(df['status'] == 'Success') & (df['vector_db_sync'] == 'No')]

In [10]:
for index, row in to_sync.iterrows():
        file_id = row['google_drive_id']
        file_name = row['file_name']
        
        text = get_text_from_postgres(file_id)

        if text:
            chunks = [text[i:i+1000] for i in range(0, len(text), 800)]
            
            print(f"File {file_name} in progress")
            
            for i, chunk in enumerate(chunks):
                vector = model.encode(chunk).tolist()
                point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{file_id}_{i}"))
                client.upsert(
                    collection_name=COLLECTION_NAME,
                    points=[
                        models.PointStruct(
                            id=point_id,
                            vector=vector,
                            payload={
                                "file_id": file_id,
                                "file_name": file_name,
                                "text": chunk,
                                "chunk_index": i
                            }
                        )
                    ]
                )

File plagiarism_check_policy.pdf in progress
File academic_integrity_policy.pdf in progress
File Plan-rozvytku-polityky-rivnosti-ta-inklyuzyvnogo-robochogo-seredovyshhe-v-UKU.pdf in progress
File Polozhennya-pro-Otsinku-efektyvnosti-pratsivnykiv-Ukrayinskogo-katolytskogo-Universytetu-1.pdf in progress
File Polozhennya-pro-pidbir-pratsivnykiv-Ukrayinskogo-katolytskogo-universytetu-1.pdf in progress
File Pravyla-vnutrishnogo-trudovogo-rozporyadku-1.pdf in progress
File Polozhennya-pro-sluzhbovi-vidryadzhennya-personalu-ZVO-UKU.pdf in progress
File Polozhennya-pro-dystantsijnu-robotu-ta-gnuchkyj-rezhym-robochogo-chasu-Ukrayinskogo-katolytskogo-universytetu-1.pdf in progress
File Polozhennya-pro-shhorichnu-vidznaku-Vykladach-roku.pdf in progress
File Poryadok-realizatsiyi-programy-rozvytku-molodyh-naukovo-pedagogichnyh-ta-pedagogichnyh-pratsivnykiv-UKU.pdf in progress
File Polozhennya-pro-poryadok-vyznannya-inozemnyh-dokumentiv-pro-osvitu-v-UKU_2020.pdf in progress
File Zminy-do-Polozhenny